In [100]:
import os
from PIL import Image
import numpy as np

In [101]:
folder_normals = "./images-blender/baked"
out_folder = "./images-blender"

In [102]:
def normalize(v):
    norm = np.linalg.norm(v, axis=2, keepdims=True)
    norm = np.maximum(norm, 1e-8)
    return v / norm

def load_normal_image(im_name):
    path = os.path.join(folder_normals, im_name)
    img = Image.open(path).convert("RGB")
    img = np.array(img).astype(np.float32) / 255.0
    # Conversion en vecteurs normaux
    n = img * 2.0 - 1.0
    n = normalize(n)
    return n

def valid_normal(n, eps=0.5):
    return np.linalg.norm(n, axis=2) > eps


In [103]:
im_names = sorted([
        f for f in os.listdir(folder_normals)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

# Charger la première image
img_mix = load_normal_image(im_names[0])

for im_name in im_names[1:]:
    img_next = load_normal_image(im_name)

    valid_mix = valid_normal(img_mix)
    valid_next = valid_normal(img_next)

    img_mix = np.where(
        valid_mix[..., None] & ~valid_next[..., None],
        img_mix,
        np.where(
            ~valid_mix[..., None] & valid_next[..., None],
            img_next,
            normalize(img_mix + img_next)
        )
    )


# Retour vers RGB
rgb_mix = (img_mix * 0.5 + 0.5) * 255.0
rgb_mix = np.clip(rgb_mix, 0, 255).astype(np.uint8)

# Sauvegarde
output_image = Image.fromarray(rgb_mix, "RGB")
output_image.show()

output_image.save(os.path.join(out_folder, "normal_merged.png"))
